# Notebook 02 — Silver → Gold

## Objetivo
Leer las tablas depuradas de Silver y materializar en el Lakehouse Gold dos tablas Delta:

- `FactSales`: hechos de ventas.
- `DimCustomer`: dimensión de clientes.

Al final se realiza una consulta de comprobación mediante un `JOIN` entre ambas tablas.

## Requisitos
- Adjuntar este notebook al Lakehouse **`LH_Gold`** antes de ejecutarlo.
- Configurar las rutas ABFSS de las tablas de Silver en la siguiente celda.
- Comprobar que las rutas y los nombres corresponden a las tablas que genera el notebook Bronze → Silver.

> Las rutas del archivo exportado contenían identificadores específicos del workspace y del Lakehouse. Se han sustituido por marcadores para evitar publicar esos identificadores en GitHub.

## 1. Configurar las rutas de origen

Sustituye los valores de ejemplo por las rutas ABFSS reales de las tablas de origen en `LH_Silver`. No publiques identificadores de tu entorno en el repositorio público.

In [ ]:
from pyspark.sql.functions import col

In [ ]:
# Rutas ABFSS de las tablas de origen en LH_Silver.
# Reemplaza cada marcador por la ruta correspondiente de tu entorno Fabric.

sales_silver_path = "<ABFSS_PATH_TO_SILVER_SALES_TABLE>"
customers_silver_path = "<ABFSS_PATH_TO_SILVER_CUSTOMERS_TABLE>"

## 2. Leer las tablas Silver

Se leen las tablas Delta desde sus rutas ABFSS y se crean los DataFrames que se utilizarán para construir las tablas Gold.

In [ ]:
sales_silver_df = (
    spark.read
    .format("delta")
    .load(sales_silver_path)
)

customers_silver_df = (
    spark.read
    .format("delta")
    .load(customers_silver_path)
)

## 3. Comprobar los datos de entrada

Contar los registros permite verificar que se han leído las tablas de origen esperadas.

In [ ]:
print("Registros de ventas Silver:", sales_silver_df.count())
print("Registros de clientes Silver:", customers_silver_df.count())

## 4. Crear la tabla de hechos `FactSales`

Se seleccionan las columnas necesarias para el análisis de ventas.

In [ ]:
fact_sales_df = sales_silver_df.select(
    "SaleId",
    "SaleDate",
    "CustomerId",
    "Product",
    "Quantity",
    "UnitPrice",
    "TotalAmount"
)

## 5. Guardar `FactSales` en Gold

La tabla se guarda en formato Delta en el Lakehouse asociado al notebook (`LH_Gold`).

> **Importante:** `overwrite` reemplaza el contenido existente de la tabla. Es adecuado para este laboratorio de carga completa, pero debe evaluarse antes de utilizarlo en un proceso productivo.

In [ ]:
(
    fact_sales_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("FactSales")
)

## 6. Crear la dimensión `DimCustomer`

Se seleccionan los atributos descriptivos de los clientes.

In [ ]:
dim_customer_df = customers_silver_df.select(
    "CustomerId",
    "CustomerName",
    "City",
    "Segment"
)

## 7. Guardar `DimCustomer` en Gold

In [ ]:
(
    dim_customer_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("DimCustomer")
)

## 8. Comprobar las tablas Gold

Leer y mostrar las tablas recién creadas.

In [ ]:
display(spark.table("FactSales"))

display(spark.table("DimCustomer"))

## 9. Probar la relación entre hechos y dimensión

Se realiza un `LEFT JOIN` utilizando `CustomerId` para enriquecer las ventas con los atributos del cliente.

Esta consulta sirve como comprobación del modelo lógico. La relación formal `1:*` se debe configurar posteriormente en el modelo semántico de Power BI.

In [ ]:
sales_analysis_df = (
    fact_sales_df.alias("f")
    .join(
        dim_customer_df.alias("d"),
        col("f.CustomerId") == col("d.CustomerId"),
        "left"
    )
    .select(
        col("f.SaleId"),
        col("f.SaleDate"),
        col("d.CustomerName"),
        col("d.City"),
        col("d.Segment"),
        col("f.Product"),
        col("f.TotalAmount")
    )
)

display(sales_analysis_df)

## Resultado esperado

Al finalizar, el Lakehouse `LH_Gold` debe contener:

- `FactSales`
- `DimCustomer`

El `JOIN` final debe mostrar las ventas enriquecidas con los datos descriptivos de los clientes.